# Un mini-GPT en PyTorch

Notebook opcional del **módulo 5** del curso (en la app: `uv run dev` y la ruta `#/modulo/bloque-transformer`). Usa modelos reales; el módulo interactivo explica la teoría con NumPy.

> **Requisitos:** desde la raíz del repositorio, `uv sync --group notebooks` y abre este notebook con `uv run --group notebooks jupyter lab`. La primera ejecución descarga los modelos de Hugging Face.

Un transformer decoder-only de caracteres con bloques pre-norm, entrenado unos cientos de pasos en CPU.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

texto = ("el transformer procesa todos los tokens en paralelo y la atencion decide de donde traer "
         "la informacion mientras la red feed forward decide que hacer con ella. ") * 40
vocab = sorted(set(texto))
stoi = {c: i for i, c in enumerate(vocab)}
datos = torch.tensor([stoi[c] for c in texto])
V, T, D, H, L = len(vocab), 64, 128, 4, 4

In [ ]:
class Bloque(nn.Module):
    def __init__(self):
        super().__init__()
        self.n1, self.n2 = nn.RMSNorm(D), nn.RMSNorm(D)
        self.att = nn.MultiheadAttention(D, H, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(D, 4 * D), nn.GELU(), nn.Linear(4 * D, D))

    def forward(self, x):
        m = torch.triu(torch.ones(x.shape[1], x.shape[1], dtype=torch.bool), 1)
        h = self.n1(x)
        x = x + self.att(h, h, h, attn_mask=m, need_weights=False)[0]
        return x + self.ffn(self.n2(x))


class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok, self.pos = nn.Embedding(V, D), nn.Embedding(T, D)
        self.bloques = nn.Sequential(*[Bloque() for _ in range(L)])
        self.norma, self.cabeza = nn.RMSNorm(D), nn.Linear(D, V)

    def forward(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.shape[1]))
        return self.cabeza(self.norma(self.bloques(x)))


modelo = MiniGPT()
print(sum(p.numel() for p in modelo.parameters()), "parámetros")

In [ ]:
opt = torch.optim.AdamW(modelo.parameters(), lr=3e-3)
for paso in range(300):
    i = torch.randint(0, len(datos) - T - 1, (32,))
    x = torch.stack([datos[j:j + T] for j in i])
    y = torch.stack([datos[j + 1:j + T + 1] for j in i])
    perdida = F.cross_entropy(modelo(x).reshape(-1, V), y.reshape(-1))
    opt.zero_grad()
    perdida.backward()
    opt.step()
    if paso % 50 == 0:
        print(paso, round(perdida.item(), 3))

In [ ]:
idx = torch.tensor([[stoi["e"]]])
for _ in range(120):
    p = F.softmax(modelo(idx[:, -T:])[0, -1], -1)
    idx = torch.cat([idx, torch.multinomial(p, 1)[None]], 1)
print("".join(vocab[i] for i in idx[0]))